In [9]:
# Instalar dependências necessárias (caso não estejam instaladas)
import subprocess, sys

packages = [
    "flask",
    "flask-cors",
    "pyttsx3",
    "requests",
    "numpy",
    "opencv-python",
    "mediapipe",
    "joblib",
    "textblob"
]

for pkg in packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", pkg, "-q", "--break-system-packages"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

print("✅ Dependências verificadas.")


✅ Dependências verificadas.


In [10]:
import os
import re
import base64
import threading
import time
import json
import subprocess

import cv2
import numpy as np
import mediapipe as mp
import joblib
import requests
import pyttsx3

from textblob import TextBlob
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS

print("✅ Imports concluídos.")


✅ Imports concluídos.


In [11]:
# ─── Caminhos dos modelos (ajustar se necessário) ───────────────────────────
MODELS_DIR = "models"
MLP_PATH    = os.path.join(MODELS_DIR, "mlp_asl_landmarks.joblib")
SCALER_PATH = os.path.join(MODELS_DIR, "scaler_asl_landmarks.joblib")
LE_PATH     = os.path.join(MODELS_DIR, "label_encoder_asl_landmarks.joblib")

# ─── Verificar existência ────────────────────────────────────────────────────
missing = [p for p in [MLP_PATH, SCALER_PATH, LE_PATH] if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        f"Modelos não encontrados: {missing}\n"
        "Execute primeiro o Grupo12_2.ipynb para treinar e guardar os modelos."
    )

mlp           = joblib.load(MLP_PATH)
scaler        = joblib.load(SCALER_PATH)
label_encoder = joblib.load(LE_PATH)

print(f"✅ Modelos carregados de '{MODELS_DIR}/'")
print(f"   Classes reconhecidas: {list(label_encoder.classes_)}")

✅ Modelos carregados de 'models/'
   Classes reconhecidas: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'del', 'space']


In [12]:
# ─── MediaPipe ───────────────────────────────────────────────────────────────
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands      = mp_hands.Hands(
    static_image_mode=True,   # True porque recebemos frames isoladas via API
    max_num_hands=1,
    min_detection_confidence=0.5
)

def normalize_landmarks(lms):
    """
    Normalização idêntica à do Grupo12_3 (63 coordenadas + 21 distâncias = 84 features).
    """
    wx, wy, wz = lms[0].x, lms[0].y, lms[0].z
    base_ids   = [5, 9, 13, 17]

    dists_base = [
        ((lms[i].x - wx)**2 + (lms[i].y - wy)**2 + (lms[i].z - wz)**2)**0.5
        for i in base_ids
    ]
    scale = float(np.mean(dists_base))
    if scale < 1e-6:
        return None

    features = []
    # 63 coordenadas normalizadas
    for lm in lms:
        features.extend([
            (lm.x - wx) / scale,
            (lm.y - wy) / scale,
            (lm.z - wz) / scale
        ])
    # 21 distâncias ao pulso
    for lm in lms:
        d = ((lm.x - wx)**2 + (lm.y - wy)**2 + (lm.z - wz)**2)**0.5
        features.append(d / scale)

    return features  # 84 valores


def predict_from_frame(frame_bgr):
    """
    Recebe um frame BGR (numpy array), devolve dict com:
      - letter: letra predita (str) ou None
      - confidence: probabilidade máxima (float)
      - hand_detected: bool
      - annotated_frame_b64: frame com landmarks desenhados (base64 JPEG)
    """
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    results   = hands.process(frame_rgb)

    letter     = None
    confidence = 0.0
    hand_det   = False

    if results.multi_hand_landmarks:
        hand_det = True
        for hand_lm in results.multi_hand_landmarks:
            # Desenhar landmarks no frame
            mp_drawing.draw_landmarks(
                frame_bgr, hand_lm, mp_hands.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 255, 120), thickness=2, circle_radius=3),
                mp_drawing.DrawingSpec(color=(255, 255, 255), thickness=2)
            )
            features = normalize_landmarks(hand_lm.landmark)
            if features is not None:
                arr   = np.array(features).reshape(1, -1)
                arr_s = scaler.transform(arr)
                pred  = mlp.predict(arr_s)
                proba = mlp.predict_proba(arr_s).max()
                letter     = label_encoder.inverse_transform(pred)[0]
                confidence = float(proba)

    # Codificar frame anotado em base64
    _, buf = cv2.imencode(".jpg", frame_bgr, [cv2.IMWRITE_JPEG_QUALITY, 80])
    frame_b64 = base64.b64encode(buf).decode("utf-8")

    return {
        "letter":               letter,
        "confidence":           confidence,
        "hand_detected":        hand_det,
        "annotated_frame_b64": frame_b64
    }

print("✅ Pipeline de reconhecimento pronta.")

✅ Pipeline de reconhecimento pronta.


I0000 00:00:1774621357.295983   33276 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1774621357.296923   35399 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) Graphics (ADL GT2)


In [13]:
# ─── Configuração Ollama + correção híbrida ────────────────────────────────
OLLAMA_URL = "http://localhost:11434/api/generate"

PREFERRED_OLLAMA_MODELS = [
    "qwen2.5:3b-instruct",
    "llama3.2:3b",
    "phi3:mini",
    "llama3"
]

COMMON_REWRITE_RULES = {
    "i want water": "I want some water, please.",
    "i want some water": "I want some water, please.",
    "i want eat": "I would like something to eat.",
    "i want to eat": "I would like something to eat.",
    "want eat": "I would like something to eat.",
    "love you": "I love you."
}


def check_ollama_available():
    """Verifica se o Ollama está a correr localmente."""
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        models = [m["name"] for m in r.json().get("models", [])]
        return True, models
    except Exception:
        return False, []


def select_ollama_model(models):
    """Escolhe o melhor modelo disponível segundo uma lista de preferência."""
    if not models:
        return None

    for preferred in PREFERRED_OLLAMA_MODELS:
        for model in models:
            if model == preferred:
                return model

    for preferred in PREFERRED_OLLAMA_MODELS:
        for model in models:
            if model.startswith(preferred):
                return model

    return models[0]


def clean_phrase_spacing(text: str) -> str:
    """Normaliza espaços redundantes."""
    return re.sub(r"\s+", " ", (text or "").strip())


def basic_spell_correct_phrase(raw_phrase: str) -> str:
    """
    Correção ortográfica determinística e 100% local.
    Excelente para casos como:
    - 'loue' -> 'love'
    - 'mch'  -> 'much'
    - 'muck' -> 'much'
    """
    cleaned = clean_phrase_spacing(raw_phrase)
    if not cleaned:
        return ""

    corrected = str(TextBlob(cleaned.lower()).correct())
    corrected = clean_phrase_spacing(corrected)
    corrected = re.sub(r"\bi\b", "I", corrected)

    raw_compact = re.sub(r"\s+", "", cleaned.lower())
    if raw_compact.startswith("ilove") and corrected.lower().startswith("love "):
        corrected = "I " + corrected[0].lower() + corrected[1:]

    if corrected:
        corrected = corrected[0].upper() + corrected[1:]

    return corrected


def apply_common_rewrite_rules(text: str):
    """
    Pequeno conjunto de reformulações úteis quando o LLM local não existe
    ou quando queremos um fallback mais inteligente do que devolver o texto bruto.
    """
    key = re.sub(r"[^a-z ]+", "", clean_phrase_spacing(text).lower())
    key = clean_phrase_spacing(key)
    return COMMON_REWRITE_RULES.get(key)


def sanitize_llm_text(text: str) -> str:
    """Limpa saídas do LLM caso venham com aspas, markdown ou frases extra."""
    if not text:
        return ""

    quote_chars = ''.join([chr(34), chr(39), ' '])

    cleaned = clean_phrase_spacing(text)
    cleaned = cleaned.strip(quote_chars)

    lines = [ln.strip(quote_chars) for ln in cleaned.splitlines() if ln.strip()]
    if lines:
        cleaned = lines[0]

    cleaned = clean_phrase_spacing(cleaned)
    cleaned = re.sub(r"\bi\b", "I", cleaned)

    if cleaned:
        cleaned = cleaned[0].upper() + cleaned[1:]

    return cleaned


def looks_like_explanation(text: str) -> bool:
    """Deteta respostas explicativas em vez de devolver só a frase."""
    lowered = text.lower()
    bad_fragments = [
        "corrected sentence",
        "here is",
        "the corrected",
        "explanation",
        "raw text",
        "output:"
    ]
    return any(fragment in lowered for fragment in bad_fragments)


def build_llm_prompt(raw_phrase: str, pre_corrected: str) -> str:
    """Prompt fortemente guiado para evitar eco da frase original."""
    return f'''
You are an offline ASL finger-spelling post-processor.

The computer vision recognizer may produce misspelled English words.
Your job is to preserve the meaning, fix spelling, and optionally make the
sentence slightly more natural.

Rules:
- Be conservative.
- Preserve meaning.
- Fix obvious spelling mistakes.
- If the sentence is telegraphic, you may rewrite it into a short natural sentence.
- Do not add new facts.
- Return STRICT JSON only.

Examples:
raw: "I loue you so mch"
pre_corrected: "I love you so much"
output: {{"corrected_text":"I love you so much","operation":"spell_only"}}

raw: "I WANT WATER"
pre_corrected: "I want water"
output: {{"corrected_text":"I want some water, please.","operation":"rewrite"}}

raw: "I WANT EAT"
pre_corrected: "I want eat"
output: {{"corrected_text":"I would like something to eat.","operation":"rewrite"}}

Process this case now:
raw: "{raw_phrase}"
pre_corrected: "{pre_corrected}"
'''.strip()


def run_ollama_rewrite(raw_phrase: str, pre_corrected: str, model: str) -> dict:
    """Chama o Ollama com output estruturado em JSON para reduzir respostas vagas."""
    payload = {
        "model": model,
        "prompt": build_llm_prompt(raw_phrase, pre_corrected),
        "system": (
            "Return only valid JSON with keys corrected_text and operation. "
            "Operation must be either spell_only or rewrite."
        ),
        "format": {
            "type": "object",
            "properties": {
                "corrected_text": {"type": "string"},
                "operation": {"type": "string"}
            },
            "required": ["corrected_text", "operation"]
        },
        "stream": False,
        "options": {
            "temperature": 0.1,
            "top_p": 0.9,
            "num_predict": 80
        },
        "keep_alive": "10m"
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=45)
    response.raise_for_status()

    data = response.json()
    raw_answer = data.get("response", "").strip()

    parsed = json.loads(raw_answer)
    corrected_text = sanitize_llm_text(parsed.get("corrected_text", ""))
    operation = parsed.get("operation", "").strip() or "spell_only"

    return {
        "corrected_text": corrected_text,
        "operation": operation,
        "raw_answer": raw_answer
    }


def llm_correct_phrase(raw_phrase: str) -> dict:
    """
    Pipeline híbrido:
      1) correção ortográfica local com TextBlob
      2) fallback por regras para alguns pedidos telegráficos comuns
      3) LLM local via Ollama para melhorar / naturalizar a frase
    """
    raw_clean = clean_phrase_spacing(raw_phrase)
    if not raw_clean:
        return {
            "corrected": "",
            "model_used": "none",
            "ollama_available": False,
            "pre_corrected": "",
            "strategy": "empty"
        }

    pre_corrected = basic_spell_correct_phrase(raw_clean)
    rule_rewrite = apply_common_rewrite_rules(pre_corrected)

    available, models = check_ollama_available()

    if not available:
        fallback_text = rule_rewrite or pre_corrected or raw_clean
        return {
            "corrected": fallback_text,
            "model_used": "fallback-local (TextBlob/rules)",
            "ollama_available": False,
            "pre_corrected": pre_corrected,
            "strategy": "local_fallback"
        }

    model = select_ollama_model(models)

    try:
        llm_result = run_ollama_rewrite(raw_clean, pre_corrected, model)
        llm_text = llm_result["corrected_text"]

        if not llm_text or looks_like_explanation(llm_text):
            final_text = rule_rewrite or pre_corrected or raw_clean
            strategy = "fallback_after_invalid_llm"
        elif llm_text.lower() == raw_clean.lower() and pre_corrected.lower() != raw_clean.lower():
            final_text = rule_rewrite or pre_corrected
            strategy = "fallback_after_llm_echo"
        else:
            final_text = llm_text
            strategy = f"llm_{llm_result['operation']}"

        return {
            "corrected": final_text,
            "model_used": model,
            "ollama_available": True,
            "pre_corrected": pre_corrected,
            "strategy": strategy
        }

    except Exception as e:
        fallback_text = rule_rewrite or pre_corrected or raw_clean
        return {
            "corrected": fallback_text,
            "model_used": f"fallback-local (erro LLM: {str(e)})",
            "ollama_available": False,
            "pre_corrected": pre_corrected,
            "strategy": "fallback_after_exception"
        }


# Teste rápido
ok, models = check_ollama_available()
if ok:
    selected = select_ollama_model(models)
    print(f"✅ Ollama disponível. Modelos: {models}")
    print(f"   Modelo preferido selecionado: {selected}")
else:
    print("⚠️  Ollama não detectado em localhost:11434.")
    print("   O servidor vai funcionar com correção local (TextBlob + regras).")

for example in ["I loue you so mch", "loue you", "I WANT WATER", "I WANT EAT", "I WANT TO DRINK", "I wrote book", "I study biology"]:
    print(f"   {example!r} -> {llm_correct_phrase(example)['corrected']!r}")


✅ Ollama disponível. Modelos: ['qwen2.5:3b-instruct', 'llama3.2:1b']
   Modelo preferido selecionado: qwen2.5:3b-instruct


W0000 00:00:1774621357.313320   35384 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774621357.325673   35394 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


   'I loue you so mch' -> 'I love you so much'
   'loue you' -> 'Love you'
   'I WANT WATER' -> 'I want water'
   'I WANT EAT' -> 'I would like to eat.'
   'I WANT TO DRINK' -> 'I want to drink'
   'I wrote book' -> 'I wrote a book'
   'I study biology' -> 'I study biology'


In [14]:
# ─── Motor TTS ───────────────────────────────────────────────────────────────
# pyttsx3 funciona 100% offline — não envia dados para servidores externos
# No Linux usa espeak/espeak-ng como backend
# No macOS usa NSSpeechSynthesizer
# No Windows usa SAPI5

# _tts_lock = threading.Lock()   # evitar chamadas TTS simultâneas

# def speak_text(text: str, rate: int = 150, volume: float = 1.0) -> dict:
#     """
#     Sintetiza o texto em voz usando pyttsx3 (processamento local).
#     Parâmetros:
#       rate   — palavras por minuto (default 150)
#       volume — 0.0 a 1.0
#     """
#     try:
#         with _tts_lock:
#             engine = pyttsx3.init()
#             engine.setProperty('rate',   rate)
#             engine.setProperty('volume', volume)
#             # Tentar selecionar voz em inglês
#             voices = engine.getProperty('voices')
#             for v in voices:
#                 if 'english' in v.name.lower() or 'en' in v.id.lower():
#                     engine.setProperty('voice', v.id)
#                     break
#             engine.say(text)
#             engine.runAndWait()
#             engine.stop()
#         return {"success": True, "text_spoken": text}
#     except Exception as e:
#         return {"success": False, "error": str(e)}







_tts_lock = threading.Lock()

def choose_english_voice(engine):
    voices = engine.getProperty("voices")

    preferred = [
        "en-gb", "english_rp", "english-us", "en-us",
        "english", "gmw/en"
    ]

    # 1) tentar correspondência direta por prioridade
    for pref in preferred:
        for v in voices:
            text = f"{getattr(v, 'id', '')} {getattr(v, 'name', '')}".lower()
            if pref in text:
                return v.id

    # 2) fallback: qualquer voz inglesa
    for v in voices:
        text = f"{getattr(v, 'id', '')} {getattr(v, 'name', '')}".lower()
        if "en" in text or "english" in text:
            return v.id

    return None


def speak_text(text: str, rate: int = 150, volume: float = 1.0) -> dict:
    try:
        with _tts_lock:
            engine = pyttsx3.init()
            engine.setProperty("rate", rate)
            engine.setProperty("volume", volume)

            voice_id = choose_english_voice(engine)
            if voice_id:
                engine.setProperty("voice", voice_id)

            engine.say(text)
            engine.runAndWait()
            engine.stop()

        return {"success": True, "text_spoken": text, "voice": voice_id}
    except Exception as e:
        return {"success": False, "error": str(e)}
    






def speak_async(text: str):
    """Lança a síntese de voz numa thread separada para não bloquear a API."""
    t = threading.Thread(target=speak_text, args=(text,), daemon=True)
    t.start()

print("✅ Motor TTS (pyttsx3) configurado.")

# Verificar vozes disponíveis
try:
    _e = pyttsx3.init()
    voices = _e.getProperty('voices')
    print(f"   Vozes disponíveis: {len(voices)}")
    for v in voices[:3]:
        print(f"     - {v.name} ({v.id})")
    _e.stop()
except Exception as ex:
    print(f"⚠️  Não foi possível inicializar TTS: {ex}")
    print("   No Linux: sudo apt-get install espeak espeak-ng")

✅ Motor TTS (pyttsx3) configurado.
   Vozes disponíveis: 109
     - Afrikaans (gmw/af)
     - Amharic (sem/am)
     - Aragonese (roa/an)


In [15]:
# ─── App Flask ─────────────────────────────────────────────────────────────
app = Flask(__name__, static_folder="web")
CORS(app)   # Permitir pedidos cross-origin (necessário para o frontend)


# ── Health check ───────────────────────────────────────────────────────────
@app.route("/api/health", methods=["GET"])
def health():
    ollama_ok, ollama_models = check_ollama_available()
    selected_model = select_ollama_model(ollama_models) if ollama_models else None

    return jsonify({
        "status": "ok",
        "models_loaded": True,
        "classes": list(label_encoder.classes_),
        "ollama": {
            "available": ollama_ok,
            "models": ollama_models,
            "selected_model": selected_model,
            "preferred_models": PREFERRED_OLLAMA_MODELS
        },
        "text_correction": "hybrid (TextBlob + rules + Ollama)",
        "tts": "pyttsx3"
    })


# ── Predição de gesto ───────────────────────────────────────────────────────
@app.route("/api/predict", methods=["POST"])
def predict():
    """
    Body JSON: { "frame": "<base64 JPEG>" }
    Resposta:  { "letter": "A", "confidence": 0.97, "hand_detected": true,
                 "annotated_frame": "<base64 JPEG com landmarks>" }
    """
    data = request.get_json(force=True)
    frame_b64 = data.get("frame", "")

    if not frame_b64:
        return jsonify({"error": "Campo 'frame' em falta."}), 400

    try:
        img_bytes = base64.b64decode(frame_b64)
        img_arr = np.frombuffer(img_bytes, dtype=np.uint8)
        frame_bgr = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
        frame_bgr = cv2.flip(frame_bgr, 1)
    except Exception as e:
        return jsonify({"error": f"Frame inválido: {str(e)}"}), 400

    result = predict_from_frame(frame_bgr)

    return jsonify({
        "letter": result["letter"],
        "confidence": result["confidence"],
        "hand_detected": result["hand_detected"],
        "annotated_frame": result["annotated_frame_b64"]
    })


# ── Correção / melhoria de frase ──────────────────────────────────────────
@app.route("/api/llm_correct", methods=["POST"])
def llm_correct():
    """
    Body JSON: { "phrase": "I WANT WEATER" }
    Resposta:  {
        "corrected": "I want some water, please.",
        "model_used": "qwen2.5:3b-instruct",
        "ollama_available": true,
        "pre_corrected": "I want water",
        "strategy": "llm_rewrite"
    }
    """
    data = request.get_json(force=True)
    phrase = data.get("phrase", "").strip()

    if not phrase:
        return jsonify({"error": "Campo 'phrase' em falta ou vazio."}), 400

    result = llm_correct_phrase(phrase)
    return jsonify(result)


# ── Síntese de voz ─────────────────────────────────────────────────────────
@app.route("/api/speak", methods=["POST"])
def speak():
    """
    Body JSON: { "text": "I'd like some water, please.", "rate": 150 }
    Resposta:  { "success": true, "text_spoken": "..." }
    Nota: síntese é feita em thread separada; a resposta é imediata.
    """
    data = request.get_json(force=True)
    text = data.get("text", "").strip()
    rate = int(data.get("rate", 150))

    if not text:
        return jsonify({"error": "Campo 'text' em falta ou vazio."}), 400

    speak_async(text)
    return jsonify({"success": True, "text_spoken": text})


# ── Servir interface web ──────────────────────────────────────────────────
@app.route("/", defaults={"path": ""})
@app.route("/<path:path>")
def serve_web(path):
    if path and os.path.exists(os.path.join("web", path)):
        return send_from_directory("web", path)
    return send_from_directory("web", "index.html")


print("✅ Rotas Flask definidas:")
print("   GET  /api/health")
print("   POST /api/predict")
print("   POST /api/llm_correct")
print("   POST /api/speak")
print("   GET  / → serve web/index.html")


✅ Rotas Flask definidas:
   GET  /api/health
   POST /api/predict
   POST /api/llm_correct
   POST /api/speak
   GET  / → serve web/index.html


In [16]:
# Criar pasta 'web' se não existir (o Grupo12_5 coloca o index.html lá)
os.makedirs("web", exist_ok=True)

PORT = 5000
print(f"🚀 A iniciar servidor em http://localhost:{PORT}")
print(f"   Interface web: http://localhost:{PORT}/")
print(f"   Health check:  http://localhost:{PORT}/api/health")
print()
print("   Para parar o servidor, interrompe o kernel (botão ■ no Jupyter).")
print()

app.run(
    host="0.0.0.0",
    port=PORT,
    debug=False,     # debug=True causaria reload automático que quebra o MediaPipe
    use_reloader=False,
    threaded=True    # necessário para requests simultâneos
)

🚀 A iniciar servidor em http://localhost:5000
   Interface web: http://localhost:5000/
   Health check:  http://localhost:5000/api/health

   Para parar o servidor, interrompe o kernel (botão ■ no Jupyter).

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.87:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [27/Mar/2026 14:23:10] "GET /api/health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Mar/2026 14:23:13] "OPTIONS /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Mar/2026 14:23:13] "POST /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Mar/2026 14:23:14] "POST /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Mar/2026 14:23:14] "POST /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Mar/2026 14:23:14] "POST /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Mar/2026 14:23:14] "POST /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Mar/2026 14:23:14] "POST /api/predict HTTP/1.1" 200 -
/home/tomas/.local/lib/python3

---
## Alternativa: Executar como Script

Em vez de correr no Jupyter, pode exportar como script e executar diretamente:

```bash
jupyter nbconvert --to script Grupo12_4.ipynb
python Grupo12_4.py
```